# 05. Machine Learning Pricing Model & Segmentation

Training Regression models (Linear Regression, Random Forest, Gradient Boosting) for pricing valuation, pricing opportunity gap analysis, and K-Means listing clustering.


In [ ]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.feature_engineering import engineer_features, compute_demand_score
from src.forecasting import train_pricing_model
from src.analysis import segment_listings_kmeans

df = pd.read_csv(project_root / 'data' / 'processed' / 'airbnb_cleaned.csv')
df = engineer_features(df)
df['demand_score'] = compute_demand_score(df)


## 1. Train Pricing Models (LR vs RF vs GB)


In [ ]:
model = train_pricing_model(df)
print('Model Evaluation Results:')
for name, res in model.results.items():
    print(f'  {name}: MAE=${res.get("mae",0):.2f}, RMSE=${res.get("rmse",0):.2f}, R2={res.get("r2",0):.4f}')
print(f'\nBest Model: {model.get_best_model_name()}')


## 2. Feature Importance


In [ ]:
best_m = model.models[model.get_best_model_name()]
fi = model.get_feature_importance(best_m, model.feature_names_)
print(fi)


## 3. Pricing Gap & Opportunity Identification


In [ ]:
df['predicted_price'] = model.predict_prices(df)
df['pricing_gap'] = model.compute_pricing_gaps(df)

underpriced = df[df['pricing_gap'] < -50]
print(f'Total Underpriced Opportunities Identified: {len(underpriced):,}')
print(underpriced[['listing_id', 'neighbourhood', 'room_type', 'price', 'predicted_price', 'pricing_gap']].head(10))


## 4. K-Means Listing Clustering


In [ ]:
clustered_df = segment_listings_kmeans(df)
print(clustered_df.groupby('cluster_name')[['price', 'estimated_annual_revenue', 'demand_score', 'estimated_occupancy_rate']].mean())
